In [ ]:
# check if the model we're using can be fine tuned
# ALL_AGENTS_MODEL_NAME

In [35]:
from app import EXIT_ADVISOR_AGENT_MODEL, ALL_AGENTS_MODEL_NAME
from app import EXIT_ADVISOR_SYSTEM_PROMPT

- We created training and test samples by iterating over each conversation and extracting an example at every candidate message, using all prior turns as conversation history and the current candidate message as latest_user_input. 

- The ground-truth label was taken from the next recruiter turn, filtering only relevant labels (continue and end). All examples were first stored in a unified intermediate format, then split into training and test sets using a label-balanced split: examples were grouped by label (end and continue), shuffled, and split separately (e.g., 80/20) to ensure both classes (end and continue) are proportionally represented in each set. 

- Finally, the training set was formatted in chat-style JSONL (including the correct assistant response) for fine-tuning, while the test set was formatted separately (without exposing the answer in the input) for consistent before-and-after evaluation of the model.

In [15]:
import json
import random
from typing import Dict, List, Tuple

def build_history_text(turns: List[Dict]) -> str:
    lines = []
    for turn in turns:
        speaker = "Recruiter" if turn["speaker"] == "recruiter" else "Candidate"
        text = turn["text"].replace("\u202f", " ")
        lines.append(f"{speaker}: {text}")
    return "\n".join(lines)


def extract_exit_examples(conversations: List[Dict]) -> List[Dict]:
    examples = []

    for conv in conversations:
        turns = conv["turns"]

        for i, turn in enumerate(turns):
            if turn["speaker"] != "candidate":
                continue

            next_recruiter_turn = None
            for j in range(i + 1, len(turns)):
                if turns[j]["speaker"] == "recruiter":
                    next_recruiter_turn = turns[j]
                    break

            if next_recruiter_turn is None:
                continue

            label = next_recruiter_turn.get("label")
            if label not in {"continue", "end"}:
                continue

            prior_turns = turns[:i]
            latest_user_input = turn["text"].replace("\u202f", " ")
            history_text = build_history_text(prior_turns)

            examples.append(
                {
                    "conversation_history": history_text,
                    "latest_user_input": latest_user_input,
                    "correct_label": label,
                }
            )

    return examples


def label_balanced_split(
    examples: List[Dict],
    train_ratio: float = 0.8,
    seed: int = 42,
) -> Tuple[List[Dict], List[Dict]]:
    continue_examples = [ex for ex in examples if ex["correct_label"] == "continue"]
    end_examples = [ex for ex in examples if ex["correct_label"] == "end"]

    rng = random.Random(seed)
    rng.shuffle(continue_examples)
    rng.shuffle(end_examples)

    def split_group(group: List[Dict]) -> Tuple[List[Dict], List[Dict]]:
        if len(group) < 2:
            raise ValueError("Each label group must contain at least 2 examples for train/test split.")
        split_idx = max(1, min(len(group) - 1, int(len(group) * train_ratio)))
        return group[:split_idx], group[split_idx:]

    cont_train, cont_test = split_group(continue_examples)
    end_train, end_test = split_group(end_examples)

    train_examples = cont_train + end_train
    test_examples = cont_test + end_test

    rng.shuffle(train_examples)
    rng.shuffle(test_examples)

    return train_examples, test_examples


def to_train_format(example: Dict) -> Dict:
    return {
        "messages": [
            {
                "role": "system",
                "content": EXIT_ADVISOR_SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": (
                    f"Conversation history:\n{example['conversation_history']}\n\n"
                    f"Latest candidate message:\n{example['latest_user_input']}"
                ),
            },
            {
                "role": "assistant",
                "content": example["correct_label"],
            },
        ]
    }


def to_test_format(example: Dict) -> Dict:
    return {
        "item": {
            "conversation_history": example["conversation_history"],
            "latest_user_input": example["latest_user_input"],
            "correct_label": example["correct_label"],
        }
    }


def write_jsonl(path: str, rows: List[Dict]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def create_exit_advisor_train_test_files(
    input_json_path: str,
    train_output_path: str,
    test_output_path: str,
    train_ratio: float = 0.8,
    seed: int = 42,
) -> Tuple[int, int, int]:
    with open(input_json_path, "r", encoding="utf-8") as f:
        conversations = json.load(f)

    examples = extract_exit_examples(conversations)

    if len(examples) < 4:
        raise ValueError("Not enough examples to create meaningful train/test files.")

    train_examples, test_examples = label_balanced_split(
        examples=examples,
        train_ratio=train_ratio,
        seed=seed,
    )

    train_rows = [to_train_format(ex) for ex in train_examples]
    test_rows = [to_test_format(ex) for ex in test_examples]

    write_jsonl(train_output_path, train_rows)
    write_jsonl(test_output_path, test_rows)

    return len(examples), len(train_rows), len(test_rows)

In [16]:
total, train_n, test_n = create_exit_advisor_train_test_files(
    input_json_path="sms_conversations.json",
    train_output_path="exit_advisor_train.jsonl",
    test_output_path="exit_advisor_test.jsonl",
    train_ratio=0.8,
    seed=42,
)

print("Total:", total)
print("Train:", train_n)
print("Test:", test_n)

Total: 25
Train: 20
Test: 5


In [17]:
from openai import OpenAI

In [18]:
from dotenv import load_dotenv
import os

load_dotenv()  # Loads variables from .env

openai_key = os.getenv("OPENAI_API_KEY")
print(openai_key[:5])  # Just to check, don't print the full key!

sk-pr


In [19]:
client = OpenAI()

In [20]:
file_obj_exit_advisor_test = client.files.create(
    file=open("exit_advisor_test.jsonl", "rb"),
    purpose="evals"
)


print(file_obj_exit_advisor_test)
print(file_obj_exit_advisor_test.id)

FileObject(id='file-Jji61GL8oj7zga1SycU9rZ', bytes=2359, created_at=1776025760, filename='exit_advisor_test.jsonl', object='file', purpose='evals', status='processed', expires_at=None, status_details=None)
file-Jji61GL8oj7zga1SycU9rZ


In [24]:
eval_obj_exit_advisor = client.evals.create(

    name="Exit Advisor Labeling",


    data_source_config={
        "type": "custom",
        "item_schema": {
            "type": "object",
            "properties": {
                "conversation_history": {"type": "string"},
                "latest_user_input": {"type": "string"},
                "correct_label": {"type": "string"},
            },
            "required": ["conversation_history", "latest_user_input", "correct_label"],
        },
        "include_sample_schema": True,


    },
    testing_criteria=[
        {
            "type": "string_check",
            "name": "Match output to human label",
            "input": "{{ sample.output_text }}",
            "operation": "eq",
            "reference": "{{ item.correct_label }}",
        }
    ],
)


print(eval_obj_exit_advisor)

EvalCreateResponse(id='eval_69dbf5143b44819196b1fbc1dd52b37d', created_at=1776022804, data_source_config=EvalCustomDataSourceConfig(schema_={'type': 'object', 'properties': {'item': {'type': 'object', 'properties': {'conversation_history': {'type': 'string'}, 'latest_user_input': {'type': 'string'}, 'correct_label': {'type': 'string'}}, 'required': ['conversation_history', 'latest_user_input', 'correct_label']}, 'sample': {'type': 'object', 'properties': {'model': {'type': 'string'}, 'choices': {'type': 'array', 'items': {'type': 'object', 'properties': {'message': {'type': 'object', 'properties': {'role': {'type': 'string', 'enum': ['assistant']}, 'content': {'type': ['string', 'array', 'null']}, 'refusal': {'type': ['boolean', 'null']}, 'tool_calls': {'type': ['array', 'null'], 'items': {'type': 'object', 'properties': {'type': {'type': 'string', 'enum': ['function']}, 'function': {'type': 'object', 'properties': {'name': {'type': 'string'}, 'arguments': {'type': 'string'}}, 'require

In [43]:
eval_obj_exit_advisor.id

'eval_69dbf5143b44819196b1fbc1dd52b37d'

In [21]:
eval_obj_exit_advisor = client.evals.retrieve('eval_69dbf5143b44819196b1fbc1dd52b37d')

In [22]:
run = client.evals.runs.create(
    eval_id=eval_obj_exit_advisor.id,
    name="exit-advisor-routing-run",
    data_source={
        "type": "completions",
        "source": {
            "type": "file_id",
            "id": file_obj_exit_advisor_test.id,
        },
        "model": EXIT_ADVISOR_AGENT_MODEL,
        "input_messages": {
            "type": "template",
            "template": [
                {
                    "role": "system",
                    "content": EXIT_ADVISOR_SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": "Conversation history:\n{{ item.conversation_history }}\n\nLatest candidate message:\n{{ item.latest_user_input }}",
                },
            ],
        },
    },
)

In [23]:
run_retrieve = client.evals.runs.retrieve(
    eval_id=eval_obj_exit_advisor.id, # YOUR_EVAL_ID
    run_id=run.id # YOUR_RUN_ID
    )


print(run_retrieve)
print(run_retrieve.status)

RunRetrieveResponse(id='evalrun_69dc00d5cd3081918b58a43404a1b65b', created_at=1776025814, data_source=CreateEvalCompletionsRunDataSource(source=SourceFileID(id='file-Jji61GL8oj7zga1SycU9rZ', type='file_id'), type='completions', input_messages=InputMessagesTemplate(template=[EasyInputMessage(content='You are an exit advisor in a recruitment conversation.\n\nYour task is to determine whether the conversation should continue or end.\n\nUse these rules:\n- Return "end" if the user clearly agreed to a specific interview date and time\n- Return "end" if the user clearly indicates they do not want to continue\n- Return "end" if the candidate is clearly not qualified and the conversation should be closed politely\n- Otherwise return "continue"\n\nGuidelines:\n- Base your decision only on the conversation history and the latest user message\n- If the user clearly confirms a proposed interview slot, return "end"\n- If the user clearly disengages (for example: "stop", "not interested", "remove me

### Accuracy

In [24]:
passed = run_retrieve.result_counts.passed
total = run_retrieve.result_counts.total

print('Accuracy')
print(len('Accuracy')*'-')
print(f'{(passed / total) :.2%}')

Accuracy
--------
60.00%


We got 60% accuracy.
We will make a fine tuning for the exit advisor model and test the fine tuned model.

#### Exit Advisor Fine Tuning

Upload the training file to OpenAI

In [25]:
exit_advisor_taining_file_obj = client.files.create(
    file=open("exit_advisor_train.jsonl", "rb"),
    purpose="fine-tune"
)

print(exit_advisor_taining_file_obj)
print(exit_advisor_taining_file_obj.id)

FileObject(id='file-6U5WiyV3MURL8JXqKzbPiH', bytes=30730, created_at=1776025947, filename='exit_advisor_train.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)
file-6U5WiyV3MURL8JXqKzbPiH


Create a Fine Tuning Job

In [26]:
exit_advisor_fine_tune = client.fine_tuning.jobs.create(
 model = EXIT_ADVISOR_AGENT_MODEL,
 training_file=exit_advisor_taining_file_obj.id,
 method={"type":"supervised"}

)


print(exit_advisor_fine_tune)

FineTuningJob(id='ftjob-EMaz1WsELU74AhDKQyA1SAls', created_at=1776025961, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto'), model='gpt-4.1-mini-2025-04-14', object='fine_tuning.job', organization_id='org-20uU6f0sgtWqgv7PHsgyNxye', result_files=[], seed=1483385512, status='validating_files', trained_tokens=None, training_file='file-6U5WiyV3MURL8JXqKzbPiH', validation_file=None, estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto'))), user_provided_suffix=None, usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None, internal_peashooter_execution=None, train_experiment_id=None, eval_experiment_id=None)


In [27]:
print(exit_advisor_fine_tune.id)

ftjob-EMaz1WsELU74AhDKQyA1SAls


In [31]:
list_fine_tune = client.fine_tuning.jobs.list()
for fine_tune_job in list(list_fine_tune):
    if fine_tune_job.id == exit_advisor_fine_tune.id:
        print (fine_tune_job.status)


succeeded


In [40]:
job = client.fine_tuning.jobs.retrieve(exit_advisor_fine_tune.id)

print(job.fine_tuned_model)

ft:gpt-4.1-mini-2025-04-14:personal::DTwSmSqk


##### Now let's test the fine tuned model

In [41]:
run_exit_model_fine_tuned = client.evals.runs.create(
    eval_id=eval_obj_exit_advisor.id,
    name="exit-advisor-routing-run",
    data_source={
        "type": "completions",
        "source": {
            "type": "file_id",
            "id": file_obj_exit_advisor_test.id,
        },
        "model": job.fine_tuned_model,
        "input_messages": {
            "type": "template",
            "template": [
                {
                    "role": "system",
                    "content": EXIT_ADVISOR_SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": "Conversation history:\n{{ item.conversation_history }}\n\nLatest candidate message:\n{{ item.latest_user_input }}",
                },
            ],
        },
    },
)

In [42]:
run_retrieve = client.evals.runs.retrieve(
    eval_id=eval_obj_exit_advisor.id, # YOUR_EVAL_ID
    run_id=run_exit_model_fine_tuned.id # YOUR_RUN_ID
    )


print(run_retrieve)
print(run_retrieve.status)

RunRetrieveResponse(id='evalrun_69dc631f6768819189235f00f481adb8', created_at=1776050975, data_source=CreateEvalCompletionsRunDataSource(source=SourceFileID(id='file-Jji61GL8oj7zga1SycU9rZ', type='file_id'), type='completions', input_messages=InputMessagesTemplate(template=[EasyInputMessage(content='You are an exit advisor in a recruitment conversation.\n\nYour task is to determine whether the conversation should continue or end.\n\nUse these rules:\n- Return "end" if the user clearly agreed to a specific interview date and time\n- Return "end" if the user clearly indicates they do not want to continue\n- Return "end" if the candidate is clearly not qualified and the conversation should be closed politely\n- Otherwise return "continue"\n\nGuidelines:\n- Base your decision only on the conversation history and the latest user message\n- If the user clearly confirms a proposed interview slot, return "end"\n- If the user clearly disengages (for example: "stop", "not interested", "remove me

In [43]:
passed = run_retrieve.result_counts.passed
total = run_retrieve.result_counts.total

print('Accuracy')
print(len('Accuracy')*'-')
print(f'{(passed / total) :.2%}')

Accuracy
--------
100.00%


Accuracy was improved from 60% to 100% thanks for the fine tuning of model.
We will use 'ft:gpt-4.1-mini-2025-04-14:personal::DTwSmSqk' model for the ExitAdvisor.
It will be defined as EXIT_ADVISOR_AGENT_FINE_TUNED_MODEL to differentiate from EXIT_ADVISOR_AGENT_MODEL which was evaludated and fine tuned above.